In [1]:
from oqd_compiler_infrastructure import Post, PrettyPrint
from oqd_core.analysis.analog.cfg import AnalogCFGBuilder
from oqd_core.analysis.analog.type_checker import AnalogTypeChecker
from oqd_core.frontend.analog import parse_analog
from oqd_core.analysis.analog.symbol_table import AnalogSymbolTableBuilder
from oqd_core.compiler.analog.passes.compile import compile_analog_circuit
from oqd_analog_emulator.interpreter import Interpreter

printer = Post(PrettyPrint())

source = """
a=2 \n b = a + 3 \n H = %I %* %X \n c = true \n d = not c \n 
e = c and d \n f = c or d \n g = a <= b \n h = a >= b \n 
i = a == b \n k = a != b \n l = a - 1 \n m = a * b \n n = 2^3 \n
"""

source = """ 
a = 2 \n b = 5 \n 
if (a > 0) { \n b = 3 } \n
if (b < 0) { \n a = 5} \n
else { \n a = 10}
if (a < 0) { \n b = 10}
"""

source = """ 
n = 5 \n
while (n > 0) { \n n = n - 1}
"""

source = """
a = [2, 3, 4]
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
"""

source = """ 
r = qreg(2)
q0 = r[0]
q1 = r[1]
initialize(q0)
evolve(-(3.14 / 4) %* %X, 1, q0)
measure(q0)
"""

source = """ 
a = sin(3.14 / 4)
b = abs(-5)
c = atan2(8, 5)
"""

circuit = parse_analog(source)
cfg = AnalogCFGBuilder().run(circuit)
checker = AnalogTypeChecker(cfg)

symbol_analysis = AnalogSymbolTableBuilder(cfg, checker.dataflow_result)
symbol_table = symbol_analysis.symbol_table

circuit, cfg = compile_analog_circuit(circuit=circuit, cfg=cfg, symbol_table=symbol_table)


In [2]:
interpreter = Interpreter(graph=cfg)
interpreter.run()
interpreter.status()

{'a': 0.706825181105366, 'b': 5, 'c': 1.0121970114513341}

In [3]:
results = interpreter.results()
results

TaskResultAnalog(class_='TaskResultAnalog', layer='analog', times=[], state=[], metrics={}, counts={}, runtime=0.0)

In [4]:
cfg.to_dict()

{0: {'register_id': 0,
  'kind': 'start',
  'stmt': {},
  'preds': [],
  'succs': [1],
  'exit_nodes': [],
  'edge_labels': {}},
 1: {'register_id': 1,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'a',
   'value': {'class_': 'MathFunc',
    'func': 'sin',
    'expr': {'class_': 'MathDiv',
     'expr1': {'class_': 'MathNum', 'value': 3.14},
     'expr2': {'class_': 'MathNum', 'value': 4}}}},
  'preds': [0],
  'succs': [2],
  'exit_nodes': [],
  'edge_labels': {}},
 2: {'register_id': 2,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'b',
   'value': {'class_': 'MathFunc',
    'func': 'abs',
    'expr': {'class_': 'MathMul',
     'expr1': {'class_': 'MathNum', 'value': -1},
     'expr2': {'class_': 'MathNum', 'value': 5}}}},
  'preds': [1],
  'succs': [3],
  'exit_nodes': [],
  'edge_labels': {}},
 3: {'register_id': 3,
  'kind': 'stmt',
  'stmt': {'class_': 'Declaration',
   'name': 'c',
   'value': {'class_': 'MathFunc',
    'func': 'atan2',
    'e